In [ ]:
# ---------------- Imports ----------------

import os
import json

import yaml
import pandas as pd


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [ ]:
# ---------------- Inputs ----------------

#INPUT_FILENAME = "ibm-rank-30k-samp2/meta-llama/Llama-3.2-3B-Instruct/20260912t000622-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/meta-llama/Llama-3.1-8B-Instruct/20260912t001205-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2-no-uncertain/meta-llama/Llama-3.1-8B-Instruct/20260920t235539-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/meta-llama/Llama-3.3-70B-Instruct/20260912t153621-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/Qwen/Qwen2.5-7B-Instruct/20260912t002447-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/Qwen/Qwen2.5-14B-Instruct/20260912t003428-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/Qwen/Qwen2.5-72B-Instruct/20260913t012204-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/mistralai/Ministral-3-8B-Instruct-2512/20260912t005409-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/mistralai/Ministral-3-14B-Instruct-2512/20260912t010437-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/mistralai/Mistral-Small-3.2-24B-Instruct-2506/20260912t150257-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/microsoft/Phi-4-mini-instruct/20260912t143117-llm-consistency.jsonl"
#INPUT_FILENAME = "ibm-rank-30k-samp2/google/gemma-4-12B-it/20260912t222101-llm-consistency.jsonl"
INPUT_FILENAME = "ibm-rank-30k-samp2/cross-encoder/stsb-roberta-large/20260922t171221-llm-consistency.jsonl"


ARGUMENT_ID = 155



In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj-store"]

INPUT_PATH = os.path.join(proj_store, "experiments", "llm-consistency", INPUT_FILENAME)

input_parts = INPUT_FILENAME.split(os.sep)

model_family = input_parts[1]
model_name = input_parts[2]

OUTPUT_FOLDER = os.path.join(proj_store, "evaluation", "llm-consistency", "comparison", model_family, model_name)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

TABLE_OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, f"argument-{ARGUMENT_ID}-scores.csv")

METADATA_OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, f"argument-{ARGUMENT_ID}-metadata.json")


In [ ]:
# ---------------- Load selected argument ----------------

rows = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        if record["argument_id"] == ARGUMENT_ID:
            rows.append(record)

if not rows:
    raise ValueError(f"Argument ID {ARGUMENT_ID} not found.")

IS_SEMANTIC_SIMILARITY = "semantic_similarity" in rows[0]



In [ ]:
# ---------------- Create matrix ----------------

argument = rows[0]["argument"]
original_topic = rows[0]["original_topic"]

if IS_SEMANTIC_SIMILARITY:
    matrix = pd.DataFrame(
        {
            "Evaluation Topic": [
                row["evaluation_topic"]
                for row in rows
            ],
            "Similarity": [
                row["semantic_similarity"]
                for row in rows
            ],
        }
    )

    matrix = matrix.sort_values(
        "Similarity", ascending=False
    ).reset_index(drop=True)

else:
    matrix = pd.DataFrame(
        {
            "Evaluation Topic": [
                row["evaluation_topic"]
                for row in rows
            ],
            "Yes": [
                row["label_probs"]["Yes"]
                for row in rows
            ],
            "No": [
                row["label_probs"]["No"]
                for row in rows
            ],
        }
    )

    matrix = matrix.sort_values(
        "Yes", ascending=False
    ).reset_index(drop=True)



In [ ]:
# ---------------- Display ----------------

print(f"Model: {model_family}/{model_name}")
print(f"Argument ID: {ARGUMENT_ID}")
print(f"Original Topic: {original_topic}")
print(f"Argument: {argument}")

display(matrix)

matrix.to_csv(TABLE_OUTPUT_PATH, index=False)

model = rows[0].get("model", rows[0].get("model_choice"))

metadata = {
    "argument_id": ARGUMENT_ID,
    "original_topic": original_topic,
    "argument": argument,
    "model": model,
    "input_file": INPUT_FILENAME,
    "semantic_similarity": IS_SEMANTIC_SIMILARITY,
}

with open(METADATA_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Saved table to: {TABLE_OUTPUT_PATH}")
print(f"Saved metadata to: {METADATA_OUTPUT_PATH}")

